# 01b — Build Vector Search Index over LEDGAR Provisions

Consumes the Delta table from `01a_load_ledgar_to_delta` and produces a Databricks
Vector Search index for the agent's semantic retrieval (RAG) tool in `02a_build_agent` /
`02b_run_agent`.

1. Build retrieval corpus from the **train split only** (keeps test split out — no leakage during eval)
2. Enable Change Data Feed (required for Delta Sync indexes)
3. Create Vector Search endpoint + Delta Sync index with managed embeddings
4. Validate with sample similarity searches

In [0]:
# Configuration Widgets
dbutils.widgets.text("catalog", "workspace", "Unity Catalog name")
dbutils.widgets.text("schema", "default", "Schema")
dbutils.widgets.text("vs_endpoint", "lexpath_vs_endpoint", "Vector Search Endpoint")
dbutils.widgets.text("embedding_endpoint", "databricks-gte-large-en", "Embedding Model Endpoint")

In [0]:
# Install Vector Search Library
%pip install -q 'databricks-vectorsearch==0.75'

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Restart Python
dbutils.library.restartPython()

In [0]:
# Read Configurations & Set Variables
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VS_ENDPOINT = dbutils.widgets.get("vs_endpoint")
EMBEDDING_ENDPOINT = dbutils.widgets.get("embedding_endpoint")

LEDGAR_TABLE = f"{CATALOG}.{SCHEMA}.ledgar_lexglue"          # written by 01a
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.ledgar_vs_source"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.ledgar_provisions_index"   # must be a 3-part UC name

print(f"Source LEDGAR table: {LEDGAR_TABLE}")
print(f"Corpus table:        {SOURCE_TABLE}")
print(f"Index:               {INDEX_NAME}")

Source LEDGAR table: workspace.default.ledgar_lexglue
Corpus table:        workspace.default.ledgar_vs_source
Index:               workspace.default.ledgar_provisions_index


In [0]:
# Build Retrieval Corpus (full train split)
from pyspark.sql import functions as F

df = spark.table(LEDGAR_TABLE)

corpus_df = (
    df.filter(F.col("split") == "train")
      # 01a stores category_label as array<string> (alias of LEDGAR 'gold'); take the single label.
      .withColumn("category", F.col("category_label").getItem(0))
      .filter(F.trim(F.col("provision_text")) != "")
      # provisions are short paragraphs; no chunking needed. Truncate defensively.
      .withColumn("provision_text", F.substring("provision_text", 1, 8000))
      .select("provision_id", "provision_text", "category", "text_length")
)

# print(f"Corpus rows (full train split): {corpus_df.count()}")
display(corpus_df.limit(10))

provision_id,provision_text,category,text_length
30000,"The Company shall (a) by 9:30 a.m. (New York City time) on the Trading Day immediately following the date hereof, issue a press release disclosing the material terms of the transactions contemplated hereby, and (b) file a Current Report on Form 8-K, including the Transaction Documents as exhibits thereto, with the Commission within the time required by the Exchange Act. From and after the issuance of such press release, the Company represents to the Purchasers that it shall have publicly disclosed all material, non-public information delivered to any of the Purchasers by the Company or any of its Subsidiaries, or any of their respective officers, directors, employees or agents in connection with the transactions contemplated by the Transaction Documents. The Company and the Purchasers shall consult with each other in issuing any other press releases with respect to the transactions contemplated hereby, and neither the Company nor the Purchasers shall issue any such press release nor otherwise make any such public statement without the prior consent of the Company, with respect to any press release of the Purchasers, or without the prior consent of the Purchasers, with respect to any press release of the Company, which consent shall not unreasonably be withheld or delayed, except if such disclosure is required by law, in which case the disclosing party shall promptly provide the other party with prior notice of such public statement or communication. Notwithstanding the foregoing, the Company shall not publicly disclose the name of the Purchaser, or include the name of the Purchaser in any filing with the Commission or any regulatory agency or Trading Market, without the prior written consent of the Purchaser, except: (a) as required by federal securities law in connection with any registration statement contemplated by this Agreement and (b) to the extent such disclosure is required by law or Trading Market regulations, in which case the Company shall provide the Purchaser with prior notice of such disclosure permitted under this clause (b).",Publicity,2080
30001,"This Agreement may be executed in two or more counterparts, each of which shall be deemed an original but all of which shall constitute but one agreement.",Counterparts,154
30002,The provision of this Section 11 shall survive the Closing or termination of this Agreement for any cause.,Survival,106
30003,"No consent, approval, order or authorization of, or registration, qualification, or filing with, any governmental authority is required on the part of such Rollover Company Member in connection with the execution, delivery and performance by such Rollover Company Member of this Agreement and the issuance, sale and delivery of the Rollover Company Units to Pioneer Parent, except (a) such filings which have been or will be made prior to the Closing and (b) any notices of sale required to be filed with the Securities and Exchange Commission under Regulation D of the Securities Act, or such post-Closing filings as may be required under applicable securities laws, which will be timely filed within the applicable periods therefor .",Consents,735
30004,"If any of the Collateral shall be sold, transferred or otherwise Disposed of by the Borrower or any Restricted Subsidiary in a transaction permitted by the Loan Documents and such Collateral shall no longer constitute or be required to be Collateral under the Loan Documents, then the Administrative Agent, at the request and sole expense of the Borrower and the applicable Restricted Subsidiary, shall promptly execute and deliver all releases or other documents reasonably necessary or desirable for the release of the Liens created by the applicable Security Instrument on such Collateral. At the request and sole expense of the Borrower, a Loan Party shall be released from its obligations under the Loan Documents in the event that all the capital stock or other Equity Inter

In [0]:
# Write Corpus Table, Enable Change Data Feed
(
    corpus_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SOURCE_TABLE)
)

spark.sql(f"ALTER TABLE {SOURCE_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"Created {SOURCE_TABLE} with Change Data Feed enabled")

Created workspace.default.ledgar_vs_source with Change Data Feed enabled


In [0]:
# Create Vector Search Endpoint
import time
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

def endpoint_exists(vsc, name):
    try:
        vsc.get_endpoint(name)
        return True
    except Exception:
        return False

if not endpoint_exists(vsc, VS_ENDPOINT):
    print(f"Creating endpoint {VS_ENDPOINT}...")
    vsc.create_endpoint(
        name=VS_ENDPOINT,
        endpoint_type="STANDARD"
    )
else:
    print(f"Endpoint {VS_ENDPOINT} already exists")

# Wait for endpoint to be ready
for attempt in range(360):
    try:
        ep = vsc.get_endpoint(VS_ENDPOINT)
        if ep.get("endpoint_status", {}).get("state") == "ONLINE":
            print(f"Endpoint {VS_ENDPOINT} is ONLINE")
            break
    except Exception as e:
        # Handle transient 503s or network errors during polling
        if "503" in str(e) or "RetryError" in str(type(e).__name__):
            print(f"Transient error (attempt {attempt+1}/360): {e}")
        else:
            raise  # Re-raise unexpected errors
    time.sleep(30)
else:
    raise TimeoutError(f"Endpoint {VS_ENDPOINT} did not become ready after 180 minutes")

/home/spark-e92e4f64-f23b-447b-bfc5-75/.ipykernel/15122/command-4628680248554533-3162486508:3: DeprecationWarning: databricks-vectorsearch is deprecated and has been renamed to databricks-ai-search. Imports under 'databricks.vector_search.*' will continue to work as a thin re-export of 'databricks.ai_search.*', but new code should switch to 'pip install databricks-ai-search' and 'from databricks.ai_search.* import ...'.
  from databricks.vector_search.client import VectorSearchClient


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
Endpoint lexpath_vs_endpoint already exists
Endpoint lexpath_vs_endpoint is ONLINE


In [0]:
# Create Delta Sync Index (idempotent - safe to re-run)
def index_exists(vsc, endpoint, name):
    try:
        vsc.get_index(endpoint_name=endpoint, index_name=name)
        return True
    except Exception:
        return False

if not index_exists(vsc, VS_ENDPOINT, INDEX_NAME):
    print(f"Creating index {INDEX_NAME}...")
    index = vsc.create_delta_sync_index(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=SOURCE_TABLE,
        pipeline_type="TRIGGERED",   # manual sync; cheaper than CONTINUOUS for static corpus
        primary_key="provision_id",
        embedding_source_column="provision_text",
        embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
    )
    print("Index creation submitted. Run Cell 9a to wait for readiness.")
else:
    print(f"Index {INDEX_NAME} already exists")
    index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
    # Check if index is ready to sync
    status = index.describe().get("status", {})
    if status.get("ready", False):
        print("Index ready — triggering re-sync")
        index.sync()
    else:
        # If index failed (OFFLINE), delete and recreate
        index_state = status.get("detailed_state", "")
        if "OFFLINE" in index_state:
            print(f"Index is offline (state: {index_state}), deleting and recreating...")
            vsc.delete_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
            time.sleep(10)  # Allow deletion to complete
            print(f"Creating index {INDEX_NAME}...")
            index = vsc.create_delta_sync_index(
                endpoint_name=VS_ENDPOINT,
                index_name=INDEX_NAME,
                source_table_name=SOURCE_TABLE,
                pipeline_type="TRIGGERED",
                primary_key="provision_id",
                embedding_source_column="provision_text",
                embedding_model_endpoint_name=EMBEDDING_ENDPOINT,
            )
        else:
            print(f"Index not ready yet (state: {status.get('message', 'unknown')}). Run Cell 9a to wait.")

# Quick status check (no long wait)
status = index.describe().get("status", {})
print(f"\nIndex status: ready={status.get('ready', False)} | {status.get('message', '')}")
if not status.get("ready", False):
    print("⏳ Index is still provisioning. Run Cell 9a to wait for completion.")

Creating index workspace.default.ledgar_provisions_index...
Index workspace.default.ledgar_provisions_index is in state PROVISIONING_INDEX. Time: 0s.
Index workspace.default.ledgar_provisions_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 31s.
Index workspace.default.ledgar_provisions_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 62s.
Index workspace.default.ledgar_provisions_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 92s.
Index workspace.default.ledgar_provisions_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 123s.
Index workspace.default.ledgar_provisions_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 154s.
Index workspace.default.ledgar_provisions_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 184s.
Index workspace.default.ledgar_provisions_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 215s.
Index workspace.default.ledgar_provisions_index is in state PROVISIONING_INITIAL_SNAPSHOT. Time: 246s.
Index workspace.default.ledga

In [0]:
# Wait for Index Ready (safe to re-run after timeout)
# This cell reconnects to the existing index and waits for it to be ready.
# If your cluster times out during the wait, just re-run this cell.

import time

# Always reconnect to the index (handles post-timeout resume)
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)

print(f"Waiting for index {INDEX_NAME} to be ready...")
print("(This can take 15-30 minutes for initial embedding generation)")

# Wait for the index using the SDK's built-in poller.
# Retry the waiter itself on transient 503 / network errors.
while True:
    try:
        index.wait_until_ready(verbose=True)
        break
    except Exception as e:
        if "503" in str(e) or "RetryError" == type(e).__name__:
            print(f"Transient error while waiting, retrying: {e}")
            time.sleep(30)
        else:
            raise  # Re-raise unexpected errors

detailed = index.describe().get("status", {})
print(f"\n✅ Index ready: {detailed.get('ready', False)} | {detailed.get('message', '')}")
print(f"You can now run Cell 11 to validate retrieval.")

Waiting for index workspace.default.ledgar_provisions_index to be ready...
(This can take 15-30 minutes for initial embedding generation)
Index workspace.default.ledgar_provisions_index is in state ONLINE_NO_PENDING_UPDATE.

✅ Index ready: True | Index creation succeeded. Check latest status: https://dbc-6e816999-3b62.cloud.databricks.com/explore/data/workspace/default/ledgar_provisions_index
You can now run Cell 11 to validate retrieval.


In [0]:
# Validate Retrieval
# Reconnect to index (in case this cell runs after a cluster restart)
index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)

# Intake-style queries
test_queries = [
    "My former employer is refusing to pay severance promised in my contract",
    "The other company broke the confidentiality terms of our agreement",
    "Which state's law applies to a dispute over our software licensing deal?",
    "My landlord terminated the lease early without notice",
]

for q in test_queries:
    results = index.similarity_search(
        query_text=q,
        columns=["provision_id", "provision_text", "category"],
        num_results=3,
    )
    rows = results.get("result", {}).get("data_array", [])
    print(f"\nQUERY: {q}")
    for r in rows:
        print(f"  [{r[2]}] (score={r[-1]:.3f}) {r[1][:120]}...")

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

QUERY: My former employer is refusing to pay severance promised in my contract
  [Releases] (score=0.663) As a condition of receiving any severance payments under this Agreement, Executive must sign and not revoke, within the ...
  [Governing Laws] (score=0.648) If the Executive believes that severance benefits are not being paid as this Agreement provides, he or she must file a c...
  [Survival] (score=0.643) The obligations of the Company or its successor to pay any Severance Payments required hereunder subsequent to the termi...
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

QUERY: The other company broke the confi

## Index Build Summary

- Corpus from the **full train split** of `{catalog}.{schema}.ledgar_lexglue` (written by `01a`) → eval in 03 stays leakage-free
- All 100 labels represented; ~60k provisions indexed
- Delta Sync index uses managed embeddings (`databricks-gte-large-en`), `TRIGGERED` sync
- `02_agent`'s retrieval tool calls `similarity_search` on this index with the client's intake description